# This notebook downloads all the trained models and tests their functionalities.

In [1]:
# Change directory to the project root.
import os
os.chdir('../src')

In [2]:
!pwd

/home/masoud/Desktop/projects/volleyball_analytics/src


In [3]:
# Download weights and check the availablity.
from ml_manager.utils.downloader import download_all_models, check_model_weights, download_from_google_drive

### Download all models in one ZIP file

In [ ]:
success = download_all_models()

In [ ]:
_ = check_model_weights()

### Check initialization of the models with the downloaded weights.

In [ ]:
# Check if the models are initialized well.
from ml_manager.ml_manager import MLManager

ml_engine = MLManager()

### Download sample video from google drive and check the results on the video.

In [ ]:
from ml_manager.utils.downloader import download_from_google_drive
from demo import run_object_detection, run_video_classification
from pathlib import Path


gid = '1kmPY_a2OZ9m484Ypx-DDuK_pE0QLxxaR'
filename = 'tokyo2020-poland-vs-iran.mp4'
output_path = 'output'

Path(output_path).mkdir(parents=True, exist_ok=True)
sample_video = Path(output_path).joinpath(filename)

if not sample_video.exists():
    success = download_from_google_drive(file_id=gid, output_path=output_path)


obj_det_output = 'output/pol-vs-irn-obj-det.mp4'
vid_cls_output = 'output/pol-vs-irn-video-classification.mp4'

# run_object_detection(video_path=filename, output_path=obj_det_output)

# run_video_classification(video_path=filename, output_path=vid_cls_output)


In [14]:
import cv2
import numpy as np
import supervision as sv
from pathlib import Path

from rich.progress import Progress
from ml_manager.ml_manager import MLManager

def run_video_classification(video_path: str, output_path: str) -> None:
    """
    Run game state classification on video every 30 frames.
    
    This function:
    - Loads the ML Manager
    - Classifies game state every 30 frames
    - Visualizes classification results on the video
    - Saves the output video
    
    Args:
        video_path: Path to input video file
        output_path: Path to save output video with visualizations
    """
    print("Initializing ML Manager...")
    ml_manager = MLManager()
    
    # Check if game state classification model is available
    if not ml_manager.is_model_available('game_state_classification'):
        raise RuntimeError("Game state classification model not available")
    
    print("Opening video...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video properties: {width}x{height}, {fps} FPS, {total_frames} frames")

    # setup output folder
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # Setup video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Classification parameters
    classification_interval = 30  # Classify every 30 frames
    frame_buffer = []
    current_game_state = "Unknown"
    current_confidence = 0.0
    
    print("Processing video frames...")
    frame_count = 0
    
    with Progress() as progress:
        task = progress.add_task(
            "[green]Processing video...",
            total=total_frames
        )
    
        while True:
            ret, frame = cap.read()

            frame_count += 1
            progress.update(task, advance=1)
            
            if not ret:
                progress.update(task, advance=30)
                break
    
            # Add frame to buffer
            frame_buffer.append(frame.copy())

            if len(frame_buffer) != 30:
                continue
            
            # Classify game state every 30 frames
            game_state_result = ml_manager.classify_game_state(frame_buffer)

            current_game_state = game_state_result.predicted_class
            current_confidence = game_state_result.confidence

            progress.update(
                task,
                description=(
                    f"[green]Frame {frame_count} | "
                    f"State: {current_game_state} | "
                    f"Conf: {current_confidence:.3f}"
                )
            )

            # Annotate frame with game state
            annotated_frame = frame.copy()
    
            state_text = f"Game State: {current_game_state}"
            confidence_text = f"Confidence: {current_confidence:.3f}"
            frame_text = f"Frame: {frame_count}"
    
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.7
            thickness = 2
    
            text_size = cv2.getTextSize(
                state_text,
                font,
                font_scale,
                thickness
            )[0]
    
            cv2.rectangle(
                annotated_frame,
                (10, 10),
                (text_size[0] + 20, 100),
                (0, 0, 0),
                -1
            )
    
            cv2.rectangle(
                annotated_frame,
                (10, 10),
                (text_size[0] + 20, 100),
                (255, 255, 255),
                2
            )
    
            cv2.putText(
                annotated_frame,
                state_text,
                (15, 35),
                font,
                font_scale,
                (255, 255, 255),
                thickness
            )
    
            cv2.putText(
                annotated_frame,
                confidence_text,
                (15, 60),
                font,
                font_scale,
                (255, 255, 255),
                thickness
            )
    
            cv2.putText(
                annotated_frame,
                frame_text,
                (15, 85),
                font,
                font_scale,
                (255, 255, 255),
                thickness
            )
    
            if current_game_state == "Play":
                color = (0, 255, 0)
            elif current_game_state == "No Play":
                color = (0, 0, 255)
            elif current_game_state == "Service":
                color = (255, 0, 255)
            else:
                color = (255, 255, 255)
    
            cv2.rectangle(
                annotated_frame,
                (10, 10),
                (text_size[0] + 20, 100),
                color,
                3
            )
    
            out.write(annotated_frame)
    
    # Cleanup
    cap.release()
    out.release()
    ml_manager.cleanup()
    
    print(f"Video classification completed. Output saved to: {output_path}")

In [15]:
gid = '1kmPY_a2OZ9m484Ypx-DDuK_pE0QLxxaR'
filename = 'tokyo2020-poland-vs-iran.mp4'
output_path = 'output'

Path(output_path).mkdir(parents=True, exist_ok=True)
sample_video = Path(output_path).joinpath(filename)

if not sample_video.exists():
    success = download_from_google_drive(file_id=gid, output_path=output_path)


obj_det_output = 'output/pol-vs-irn-obj-det.mp4'
vid_cls_output = 'output/pol-vs-irn-video-classification.mp4'


run_video_classification(video_path=filename, output_path=vid_cls_output)

Frame 30 | State: no-play | Conf: 1.000 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━  88% 0:00:01

Video classification completed. Output saved to: output/pol-vs-irn-video-classification.mp4


In [ ]:
# Testing gamestate classifier.
import numpy as np

frame = np.ones((224, 224, 3), dtype=np.uint8)
frames = [frame.copy() for _ in range(16)]


ml_engine.classify_game_state(frames)